# Notebook 008 — Extract Machine Learning Features

## Objective

In this notebook we transform each detected astronomical source into a set of
numerical features suitable for machine learning.

Rather than working directly with image pixels, we extract measurable properties
such as brightness, size, shape, and position. These measurements form a
feature table in which every row represents one astronomical object and every
column represents a measurable characteristic.

This notebook represents the transition from image processing to machine
learning and data science.

In [1]:
# Cell 2 - Import Libraries ===================================

import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from photutils.background import Background2D, MedianBackground
from photutils.segmentation import (
    detect_sources,
    SourceCatalog
)
from astropy.visualization import ImageNormalize
from astropy.visualization import PercentileInterval

In [2]:
# Cell 3 - Load FITS Image ====================================

filename = "../data/raw/mastDownload/HST/ib5x1fetq/ib5x1fetq_flt.fits"

with fits.open(filename) as hdul:
    science = hdul["SCI"].data

print("Image shape :", science.shape)
print("Data type   :", science.dtype)

Image shape : (1014, 1014)
Data type   : >f4


In [3]:
# Cell 4

from astropy.table import Table
from pathlib import Path

catalog = Table.read(
    Path("../data/catalogs/source_catalog.ecsv"),
    format="ascii.ecsv"
)

print(f"Loaded catalog with {len(catalog)} sources.")

Loaded catalog with 95 sources.


In [4]:
# Cell 5 - Examine Available Features ==========================

print(f"Number of sources : {len(catalog)}")
print(f"Number of measured features : {len(catalog.colnames)}")

print("\nAvailable catalog columns:\n")

for name in catalog.colnames:
    print(name)

Number of sources : 95
Number of measured features : 19

Available catalog columns:

label
x_centroid
y_centroid
sky_centroid
bbox_xmin
bbox_xmax
bbox_ymin
bbox_ymax
area
semimajor_axis
semiminor_axis
orientation
eccentricity
min_value
max_value
segment_flux
segment_flux_err
kron_flux
kron_flux_err


In [5]:
# Cell 6 - Select Features for Machine Learning ================

feature_columns = [
    "segment_flux",
    "area",
    "semimajor_axis",
    "semiminor_axis",
    "eccentricity",
    "orientation",
]

features = catalog[feature_columns]

features[:10]

segment_flux,area,semimajor_axis,semiminor_axis,eccentricity,orientation
,pix2,pix,pix,,deg
float64,float64,float64,float64,float64,float64
103.97923684120178,24.0,2.1532985095548427,1.4546436878746936,0.7373214628932244,59.68625869106634
19.497517228126526,16.0,1.0425375485147124,0.8473819282730207,0.5825329240047639,30.763603254853948
31.58135676383972,16.0,0.8541513400365031,0.7580456582502352,0.4608385055033623,277.7009772993434
6.465743601322174,6.0,0.6440643877584695,0.5514238892637224,0.516706370668758,294.4995327788689
24.018874406814575,32.0,2.2941649471403283,1.4005674373639603,0.7920230641969821,303.90657994640236
12.340121269226074,15.0,1.063488324717736,0.9223128857493856,0.497868595189019,271.8012506450832
11.399610340595245,13.0,1.0771237638461464,0.7989851718607586,0.6706470897789912,45.38350344446989
6.043212354183197,10.0,1.13510003434925,0.6721785777757441,0.805808924882241,318.5683560480159


## Machine Learning Features

The following physical measurements were selected from the astronomical catalog
to describe each detected source. Together, these features form the input
(feature matrix) for future machine learning algorithms.

| Feature | Physical Meaning | Why It Is Useful for Machine Learning |
|---------|------------------|----------------------------------------|
| **segment_flux** | Total measured brightness of the source | Distinguishes bright objects from faint ones and is a fundamental measure of luminosity. |
| **area** | Number of pixels belonging to the detected source | Separates compact objects from extended galaxies and provides an estimate of apparent size. |
| **semimajor_axis** | Length of the source's major axis | Describes the overall size and extent of the object. |
| **semiminor_axis** | Length of the source's minor axis | Complements the major axis and helps characterize object shape. |
| **eccentricity** | Degree to which the object is elongated | Differentiates nearly circular objects from elongated galaxies or blended sources. |
| **orientation** | Position angle of the major axis (degrees) | Records the object's orientation on the detector and provides additional morphological information. |

Together, these measurements provide a numerical description of each
astronomical object that can be used for clustering, classification, and other
machine learning techniques.

In [6]:
# Cell 8 - Engineer Additional Features ========================

features = features.copy()

# Shape descriptor
features["axis_ratio"] = (
    features["semimajor_axis"] /
    features["semiminor_axis"]
)

# Brightness density
features["flux_per_pixel"] = (
    features["segment_flux"] /
    features["area"]
)

print("Derived features added.")

features[:10]

Derived features added.


segment_flux,area,semimajor_axis,semiminor_axis,eccentricity,orientation,axis_ratio,flux_per_pixel
,pix2,pix,pix,,deg,pix,
float64,float64,float64,float64,float64,float64,float64,float64
103.97923684120178,24.0,2.1532985095548427,1.4546436878746936,0.7373214628932244,59.68625869106634,1.4802927531352494,4.332468201716741
19.497517228126526,16.0,1.0425375485147124,0.8473819282730207,0.5825329240047639,30.763603254853948,1.2303042037247858,1.2185948267579079
31.58135676383972,16.0,0.8541513400365031,0.7580456582502352,0.4608385055033623,277.7009772993434,1.1267808617334536,1.9738347977399826
6.465743601322174,6.0,0.6440643877584695,0.5514238892637224,0.516706370668758,294.4995327788689,1.1680023305091904,1.0776239335536957
24.018874406814575,32.0,2.2941649471403283,1.4005674373639603,0.7920230641969821,303.90657994640236,1.6380253359725598,0.7505898252129555
12.340121269226074,15.0,1.063488324717736,0.9223128857493856,0.497868595189019,271.8012506450832,1.1530667533216175,0.8226747512817383
11.399610340595245,13.0,1.0771237638461464,0.7989851718607586,0.6706470897789912,45.38350344446989,1.3481148358956776,0.8768931031227112
6.043212354183197,10.0,1.13510003434925,0.6721785777757441,0.805808924882241,318.5683560480159,1.6886882026281242,0.6043212354183197


In [7]:
# Cell 9 - Summary of Machine Learning Features ================

features.info()

<Table length=95>
     name       dtype  unit
-------------- ------- ----
  segment_flux float64     
          area float64 pix2
semimajor_axis float64  pix
semiminor_axis float64  pix
  eccentricity float64     
   orientation float64  deg
    axis_ratio float64  pix
flux_per_pixel float64     


In [8]:
features[:10]

segment_flux,area,semimajor_axis,semiminor_axis,eccentricity,orientation,axis_ratio,flux_per_pixel
,pix2,pix,pix,,deg,pix,
float64,float64,float64,float64,float64,float64,float64,float64
103.97923684120178,24.0,2.1532985095548427,1.4546436878746936,0.7373214628932244,59.68625869106634,1.4802927531352494,4.332468201716741
19.497517228126526,16.0,1.0425375485147124,0.8473819282730207,0.5825329240047639,30.763603254853948,1.2303042037247858,1.2185948267579079
31.58135676383972,16.0,0.8541513400365031,0.7580456582502352,0.4608385055033623,277.7009772993434,1.1267808617334536,1.9738347977399826
6.465743601322174,6.0,0.6440643877584695,0.5514238892637224,0.516706370668758,294.4995327788689,1.1680023305091904,1.0776239335536957
24.018874406814575,32.0,2.2941649471403283,1.4005674373639603,0.7920230641969821,303.90657994640236,1.6380253359725598,0.7505898252129555
12.340121269226074,15.0,1.063488324717736,0.9223128857493856,0.497868595189019,271.8012506450832,1.1530667533216175,0.8226747512817383
11.399610340595245,13.0,1.0771237638461464,0.7989851718607586,0.6706470897789912,45.38350344446989,1.3481148358956776,0.8768931031227112
6.043212354183197,10.0,1.13510003434925,0.6721785777757441,0.805808924882241,318.5683560480159,1.6886882026281242,0.6043212354183197


In [9]:
print(f"\nNumber of objects : {len(features)}")
print(f"Number of features: {len(features.colnames)}")


Number of objects : 95
Number of features: 8


In [10]:
# Cell 10 - Save Machine Learning Features =====================

output_file = Path("../data/catalogs/ml_features.ecsv")

features.write(
    output_file,
    format="ascii.ecsv",
    overwrite=True
)

print(f"Machine learning feature table saved to:\n{output_file}")

Machine learning feature table saved to:
../data/catalogs/ml_features.ecsv


# Discussion
In this notebook, the astronomical catalog was transformed into a machine learning feature table. Each detected object is now represented by a numerical vector describing its brightness, size, and morphology. This feature matrix forms the basis for future clustering and classification algorithms.

In [11]:
# Cell 11 - Notebook Status ==================================

from datetime import datetime

print("=" * 60)
print("Notebook Status")
print("=" * 60)

print("Status    : PASS")
print("Notebook  : 08_extract_machine_learning_features.ipynb")
print("Completed :", datetime.now().strftime("%Y-%m-%d %H:%M"))

Notebook Status
Status    : PASS
Notebook  : 08_extract_machine_learning_features.ipynb
Completed : 2026-07-27 15:48
